# Basic Pandas Operations with a Kaggle Dataset

This notebook uses Kaggle's [Online Food Dataset](https://www.kaggle.com/datasets/sudarshan24byte/online-food-dataset) by Sudarshan Trifaley (MIT License). The CSV is included beside this notebook as `onlinefoods.csv`.

The dataset contains survey responses about online food ordering. We will load it, inspect and clean it, select and filter rows, check data quality, summarize categories, and sort results. Pandas is the only library needed.

In [1]:
from pathlib import Path
import pandas as pd

# This works whether the notebook is run from the repository root or Pandas folder.
data_path = Path("onlinefoods.csv")
if not data_path.exists():
    data_path = Path("Pandas") / "onlinefoods.csv"

df_raw = pd.read_csv(data_path)
df = df_raw.copy()

print(f"Loaded {len(df):,} rows from {data_path}")

Loaded 388 rows from onlinefoods.csv


## 1. Preview and inspect the dataset

`head()` and `tail()` show sample rows. `shape`, `columns`, and `info()` describe the structure; `describe()` summarizes the numeric columns.

In [2]:
display(df.head())
display(df.tail(3))
print("Shape (rows, columns):", df.shape)
print("Column names:", df.columns.tolist())
df.info()

,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,Output,Feedback,Unnamed: 12
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,Yes,Positive,Yes
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,Yes,Positive,Yes
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,Yes,Negative,Yes
3,22,Female,Single,Student,No Income,Graduate,6,12.9473,77.5616,560019,Yes,Positive,Yes
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,12.9850,77.5533,560010,Yes,Positive,Yes


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,Output,Feedback,Unnamed: 12
385,22,Female,Single,Student,No Income,Post Graduate,5,12.9850,77.5533,560010,Yes,Positive,Yes
386,23,Male,Single,Student,Below Rs.10000,Post Graduate,2,12.9770,77.5773,560009,Yes,Positive,Yes
387,23,Male,Single,Student,No Income,Post Graduate,5,12.8988,77.5764,560078,Yes,Positive,Yes


Shape (rows, columns): (388, 13)
Column names: ['Age', 'Gender', 'Marital Status', 'Occupation', 'Monthly Income', 'Educational Qualifications', 'Family size', 'latitude', 'longitude', 'Pin code', 'Output', 'Feedback', 'Unnamed: 12']
<class 'pandas.DataFrame'>
RangeIndex: 388 entries, 0 to 387
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         388 non-null    int64  
 1   Gender                      388 non-null    str    
 2   Marital Status              388 non-null    str    
 3   Occupation                  388 non-null    str    
 4   Monthly Income              388 non-null    str    
 5   Educational Qualifications  388 non-null    str    
 6   Family size                 388 non-null    int64  
 7   latitude                    388 non-null    float64
 8   longitude                   388 non-null    float64
 9   Pin code                    388 non-null

In [ ]:
df.describe()

## 2. Clean the data

The CSV has an extra final column without a header. Its meaning is not documented, so we leave it out of the analysis rather than guessing what it represents. We also trim accidental spaces from text values (for example, `"Negative "` in the feedback column).

In [ ]:
unnamed_columns = df.columns[df.columns.str.startswith("Unnamed")].tolist()
df = df.drop(columns=unnamed_columns)

text_columns = df.select_dtypes(include=["object", "string"]).columns
df[text_columns] = df[text_columns].apply(lambda column: column.str.strip())

print("Removed unnamed columns:", unnamed_columns)
print("Cleaned shape:", df.shape)
print("Feedback categories:", df["Feedback"].unique())

## 3. Select columns and rows

Use a list of column names to select multiple columns. `.loc` selects by labels; `.iloc` selects by integer positions.

In [ ]:
display(df[["Age", "Occupation", "Output", "Feedback"]].head())

# Label-based selection: rows 0 through 4, inclusive.
display(df.loc[0:4, ["Age", "Gender", "Occupation"]])

# Position-based selection: first 5 rows and first 3 columns.
display(df.iloc[0:5, 0:3])

## 4. Filter rows

Combine conditions with `&` (and) or `|` (or). Put parentheses around each condition.

In [ ]:
female_customers_25_plus = df.loc[
    (df["Gender"] == "Female") & (df["Age"] >= 25),
    ["Age", "Gender", "Occupation", "Output"],
]
display(female_customers_25_plus.head(10))
print("Matching rows:", len(female_customers_25_plus))

## 5. Check missing values and duplicate rows

These checks help decide whether more cleaning is needed. We report duplicate rows but do not automatically remove them, since repeated responses may be meaningful.

In [ ]:
print("Missing values in each column:")
display(df.isna().sum())

print("Exact duplicate rows:", df.duplicated().sum())

## 6. Count and summarize categories

`value_counts()` counts category values. `groupby()` lets us calculate a summary separately for each occupation.

In [ ]:
print("Online food ordering responses:")
display(df["Output"].value_counts())

print("Average age by occupation:")
display(
    df.groupby("Occupation")["Age"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
)

## 7. Sort rows

`sort_values()` orders rows by one or more columns. Here, we display the oldest respondents first.

In [ ]:
display(
    df.sort_values("Age", ascending=False)[
        ["Age", "Gender", "Occupation", "Output", "Feedback"]
    ].head(10)
)